In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

In [2]:
df_raw = pd.read_excel("Dump Dataset.xlsx")
print("Raw shape:", df_raw.shape)
df_raw.head()

Raw shape: (264171, 27)


,Merchant Name,Courier Partner,Channel Name,Payment Mode,Zone,Created at,Status,Pickup Pincode,Pickup City,Pickup State,Pickup Country,Drop Pincode,Drop City,Drop State,Drop Country,Total Quantity,Shipment Length,Shipment Breadth,Shipment Height,Shipment Weight,Invoice Date,COD Value,Invoice Value,Order Date,Delivery Date,Min Tat,Max Tat
0,Client 1,ATS Surface,WMS,COD,ROI,2025-10-01 23:54:42,CANCELLED_ORDER,410501,Pune,Maharashtra,India,758035,Kendujhar,Odisha,IN,1,20.0000,10.0000,10.0000,1000.0000,2025-10-01 23:54:42,575.0000,575.0000,2025-10-01,,5,7
1,Client 2,Delhivery Surface,WMS,PREPAID,ROI,2025-10-01 23:53:04,DELIVERED,201301,Noida,Uttar Pradesh,IN,732207,Kaliachak,West Bengal,IN,1,5.0000,5.0000,5.0000,500.0000,2025-10-01 23:53:04,0.0000,303.3600,2025-10-01,2025-10-09 16:59:21,5,6
2,Client 3,ATS Surface,ATS_B2C_SURFACE,PREPAID,ROI,2025-10-01 23:49:22,CANCELLED_ORDER,382424,Ahmedabad,Gujrat,IN,410221,MAHARASHTRA,MAHARASHTRA,IN,1,10.0000,5.0000,5.0000,100.0000,2025-10-01 23:49:22,0.0000,199.0000,2025-10-01,,5,7
3,Client 2,Delhivery Surface,WMS,PREPAID,LOCAL,2025-10-01 23:48:49,DELIVERED,201301,Noida,Uttar Pradesh,IN,110024,Delhi Ncr,Haryana,IN,1,5.0000,5.0000,5.0000,300.0000,2025-10-01 23:48:49,0.0000,314.3700,2025-10-01,2025-10-05 12:36:07,1,2
4,Client 3,ATS Surface,ATS_B2C_SURFACE,PREPAID,ROI,2025-10-01 23:35:06,CANCELLED_ORDER,382424,Ahmedabad,Gujrat,IN,410221,MAHARASHTRA,MAHARASHTRA,IN,1,10.0000,5.0000,5.0000,100.0000,2025-10-01 23:35:06,0.0000,199.0000,2025-10-01,,5,7


In [3]:
df = df_raw.copy()

df["delivery_completed"] = (
    df["Status"]
    .astype(str)
    .str.upper()
    .eq("DELIVERED")
)


In [4]:
df["delivery_completed"].value_counts()


delivery_completed
True     217137
False     47034
Name: count, dtype: int64

In [5]:
df["order_date"] = pd.to_datetime(df["Created at"], errors="coerce")
df["delivery_date"] = pd.to_datetime(df["Delivery Date"], errors="coerce")

df = df[
    df["order_date"].notna() &
    df["delivery_date"].notna()
].copy()


C:\Users\bharg\AppData\Local\Temp\ipykernel_2728\1075928510.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["delivery_date"] = pd.to_datetime(df["Delivery Date"], errors="coerce")


In [6]:
df["actual_delivery_days"] = (
    df["delivery_date"] - df["order_date"]
).dt.days

df = df[df["actual_delivery_days"] >= 0].copy()


In [7]:
df["actual_delivery_days"].describe()


count   217138.0000
mean         3.7548
std          2.5549
min          0.0000
25%          2.0000
50%          3.0000
75%          5.0000
max         63.0000
Name: actual_delivery_days, dtype: float64

In [8]:
df["Max Tat"] = (
    df["Max Tat"]
    .astype(str)
    .str.strip()
)

df["Max Tat"] = pd.to_numeric(df["Max Tat"], errors="coerce")

before = len(df)
df = df[df["Max Tat"].notna()].copy()
after = len(df)

print("Dropped due to missing Max Tat:", before - after)


Dropped due to missing Max Tat: 3329


In [9]:
df["delayed"] = (df["actual_delivery_days"] > df["Max Tat"]).astype(int)

df["delayed"].value_counts(normalize=True)


delayed
0   0.7978
1   0.2022
Name: proportion, dtype: float64

In [20]:
df["interstate_flag"] = (
    df["Pickup State"].astype(str).str.upper() !=
    df["Drop State"].astype(str).str.upper()
).astype(int)


In [11]:
# Clean basic categoricals (consistent categories)
for c in ["Courier Partner", "Pickup State", "Drop State", "Payment Mode", "Zone"]:
    if c in df.columns:
        df[c] = df[c].astype(str).str.upper().str.strip().replace({"NAN": np.nan, "": np.nan})

# Lane (state-to-state)
df["state_lane"] = (
    df["Pickup State"].astype(str).str.upper().str.strip() +
    "__" +
    df["Drop State"].astype(str).str.upper().str.strip()
)

# Courier x Zone interaction (if Zone exists)
if "Zone" in df.columns:
    df["courier_zone"] = (
        df["Courier Partner"].astype(str).str.upper().str.strip() +
        "__" +
        df["Zone"].astype(str).str.upper().str.strip()
    )
else:
    df["courier_zone"] = (
        df["Courier Partner"].astype(str).str.upper().str.strip() + "__" + "NO_ZONE"
    )

# COD flag and COD ratio
df["is_cod"] = df["Payment Mode"].astype(str).str.upper().str.contains("COD", na=False).astype(int)

eps = 1e-6
df["cod_ratio"] = (df["COD Value"] / (df["Invoice Value"] + eps)).clip(0, 1)

# Promise window
df["promise_window"] = (df["Max Tat"] - pd.to_numeric(df.get("Min Tat", np.nan), errors="coerce")).clip(lower=0)

# Volumetric & chargeable weight proxies
df["volumetric_weight_proxy"] = (df["shipment_volume"] ** (1/3))
df["chargeable_weight_proxy"] = np.maximum(df["Shipment Weight"], df["volumetric_weight_proxy"])

# Time features from order_date (created-at)
df["order_dow"] = df["order_date"].dt.day_name()
df["order_month"] = df["order_date"].dt.month


In [10]:
eps = 1e-6
df["shipment_volume"] = (
    df["Shipment Length"] *
    df["Shipment Breadth"] *
    df["Shipment Height"]
)

df["density"] = df["Shipment Weight"] / (df["shipment_volume"] + eps)


Zones were excluded because they are deterministic transformations of pickup and drop states and do not add independent predictive signal when state-level geography is already present.

In [12]:
LEAKAGE_COLS = [
    "Delivery Date", "delivery_date",
    "actual_delivery_days",
    "Status",
    "delivery_completed",
    "Created at", "order_date",   # optional to drop raw timestamps
    "delayed"  # target kept separately
]

df_cb = df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns], errors="ignore").copy()

CB_NUMERIC = [
    "Shipment Weight",
    "shipment_volume",
    "Total Quantity",
    "Max Tat",
    "density",
    "chargeable_weight_proxy",
    "cod_ratio",
    "promise_window",
    "order_month"
]

CB_CATEGORICAL = [
    "Courier Partner",
    "Pickup State",
    "Drop State",
    "Payment Mode",
    "Zone",
    "state_lane",
    "courier_zone",
    "order_dow"
]

CB_BINARY = ["interstate_flag", "is_cod"]

TARGET = "delayed"

# Keep only columns that exist (robust)
CB_NUMERIC = [c for c in CB_NUMERIC if c in df.columns]
CB_CATEGORICAL = [c for c in CB_CATEGORICAL if c in df.columns]
CB_BINARY = [c for c in CB_BINARY if c in df.columns]

df_cb = df[CB_NUMERIC + CB_CATEGORICAL + CB_BINARY + [TARGET]].copy()

# Minimal imputation (CatBoost can handle missing, but this keeps things clean)
for c in CB_CATEGORICAL:
    df_cb[c] = df_cb[c].fillna("UNKNOWN")

# Numeric: keep NaNs or impute; I recommend median to keep NN parity later
for c in CB_NUMERIC:
    df_cb[c] = df_cb[c].astype(float)
    df_cb[c] = df_cb[c].fillna(df_cb[c].median())

df_cb[TARGET] = df_cb[TARGET].astype(int)

print("CatBoost-ready shape:", df_cb.shape)
print(df_cb[TARGET].value_counts(normalize=True))


CatBoost-ready shape: (213809, 19)
delayed
0   0.7978
1   0.2022
Name: proportion, dtype: float64


In [13]:
df_cb.to_csv("model_ready_catboost.csv", index=False)


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Use the same feature lists but NN needs numeric matrix
X = df_cb.drop(columns=[TARGET]).copy()
y = df_cb[TARGET].copy()

num_cols = [c for c in CB_NUMERIC + CB_BINARY if c in X.columns]
cat_cols = [c for c in CB_CATEGORICAL if c in X.columns]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ],
    remainder="drop"
)

X_nn = preprocess.fit_transform(X)

# Build feature names (for saving as a dataframe)
ohe = preprocess.named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = list(ohe.get_feature_names_out(cat_cols))
feature_names = num_cols + cat_feature_names

X_nn_df = pd.DataFrame(X_nn.toarray() if hasattr(X_nn, "toarray") else X_nn, columns=feature_names)
X_nn_df[TARGET] = y.values

print("NN-ready shape:", X_nn_df.shape)
print(X_nn_df[TARGET].value_counts(normalize=True))


NN-ready shape: (213809, 1642)
delayed
0   0.7978
1   0.2022
Name: proportion, dtype: float64


In [15]:
X_nn_df.to_csv("model_ready_nn.csv", index=False)


In [16]:
df_tree = df.copy()

In [17]:
df_tree["shipment_volume"] = (
    df_tree["Shipment Length"] *
    df_tree["Shipment Breadth"] *
    df_tree["Shipment Height"]
)

df_tree["density"] = df_tree["Shipment Weight"] / (df_tree["shipment_volume"] + eps)

# COD ratio
df_tree["cod_ratio"] = (
    df_tree["COD Value"] / (df_tree["Invoice Value"] + eps)
).clip(0, 1)

# Promise window
df_tree["promise_window"] = (
    df_tree["Max Tat"] - pd.to_numeric(df_tree["Min Tat"], errors="coerce")
).clip(lower=0)

# Chargeable weight
df_tree["volumetric_weight_proxy"] = df_tree["shipment_volume"] ** (1/3)
df_tree["chargeable_weight_proxy"] = np.maximum(
    df_tree["Shipment Weight"],
    df_tree["volumetric_weight_proxy"]
)

# Time
df_tree["order_dow"] = df_tree["order_date"].dt.day_name()
df_tree["order_month"] = df_tree["order_date"].dt.month

In [18]:
at_cols = [
    "Courier Partner",
    "Zone",
    "Payment Mode",
    "Pickup State",
    "Drop State"
]

for c in cat_cols:
    if c in df_tree.columns:
        df_tree[c] = (
            df_tree[c]
            .astype(str)
            .str.upper()
            .str.strip()
            .replace({"NAN": np.nan, "": np.nan})
            .fillna("UNKNOWN")
        )

In [19]:
df_tree["state_lane"] = (
    df_tree["Pickup State"] + "__" + df_tree["Drop State"]
)

TOP_LANES = 40
top_lanes = df_tree["state_lane"].value_counts().head(TOP_LANES).index
df_tree["state_lane"] = df_tree["state_lane"].where(
    df_tree["state_lane"].isin(top_lanes),
    "OTHER"
)

In [20]:
df_tree["courier_zone"] = (
    df_tree["Courier Partner"] + "__" + df_tree["Zone"]
)

TOP_CZ = 50
top_cz = df_tree["courier_zone"].value_counts().head(TOP_CZ).index
df_tree["courier_zone"] = df_tree["courier_zone"].where(
    df_tree["courier_zone"].isin(top_cz),
    "OTHER"
)

In [21]:
TREE_NUM = [
    "Shipment Weight",
    "shipment_volume",
    "Total Quantity",
    "Max Tat",
    "density",
    "chargeable_weight_proxy",
    "cod_ratio",
    "promise_window",
    "order_month"
]

TREE_CAT = [
    "Zone",
    "Courier Partner",
    "Payment Mode",
    "Pickup State",
    "Drop State",
    "state_lane",
    "courier_zone",
    "order_dow"
]

TARGET = "delayed"

TREE_NUM = [c for c in TREE_NUM if c in df_tree.columns]
TREE_CAT = [c for c in TREE_CAT if c in df_tree.columns]

df_tree_final = df_tree[TREE_NUM + TREE_CAT + [TARGET]].copy()


In [22]:
for c in TREE_NUM:
    df_tree_final[c] = pd.to_numeric(df_tree_final[c], errors="coerce")
    df_tree_final[c] = df_tree_final[c].fillna(df_tree_final[c].median())


In [23]:
df_tree_ohe = pd.get_dummies(
    df_tree_final,
    columns=TREE_CAT,
    drop_first=True
)


In [24]:
df_tree_ohe.to_csv("model_ready_tree.csv", index=False)

print("Tree-ready dataset shape:", df_tree_ohe.shape)
print("Target distribution:")
print(df_tree_ohe[TARGET].value_counts(normalize=True) * 100)


Tree-ready dataset shape: (213809, 434)
Target distribution:
delayed
0   79.7754
1   20.2246
Name: proportion, dtype: float64
